In [ ]:
import os
import re
import math
import pandas as pd
from pathlib import Path
from pyproj import Transformer

def highlight_close_distances(val):
    """
    Returns a CSS string to style cells green if the distance is less than 4 meters.
    """
    if isinstance(val, (int, float)) and val < 2:
        return 'background-color: #c8e6c9; color: #1b5e20; font-weight: bold;'  # Soft green background with dark green text
    return ''

# Wezep

In [2]:
# Borehole data Wezep
borehole_data = {
    'Borehole_ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'X': [196014, 195980, 195998, 195988, 196005, 195986, 195999, 195979, 196001, 195992, 195994, 195985],
    'Y': [497413, 497435, 497401, 497411, 497415, 497440, 497410, 497424, 497422, 497434, 497418, 497429]
}

df_boreholes = pd.DataFrame(borehole_data)

# Change this to the path where your .gef files are stored
gef_folder_path = r"C:\AA_Thesis\CPT_Measurements\Wezep"  

cpt_coordinates = {}

# Process each file in the folder
for filename in os.listdir(gef_folder_path):
    if filename.endswith('.gef'):
        # Extract CPT ID using regex (matches everything between the last '_' and '.gef')
        match_id = re.search(r'_([A-Za-z0-9]+)\.gef$', filename)
        if not match_id:
            continue
        cpt_id = match_id.group(1)
        
        filepath = os.path.join(gef_folder_path, filename)
        
        # Read file to find the #XYID line
        with open(filepath, 'r', encoding='latin-1') as file:
            for line in file:
                if line.startswith('#XYID='):
                    # Clean line and split by commas
                    parts = line.replace('#XYID=', '').strip().split(',')
                    if len(parts) >= 3:
                        try:
                            # 2nd value is X, 3rd value is Y
                            cpt_x = float(parts[1].strip())
                            cpt_y = float(parts[2].strip())
                            cpt_coordinates[cpt_id] = (cpt_x, cpt_y)
                        except ValueError:
                            print(f"Could not parse coordinates in file: {filename}")
                    break

# Initialize the final distance matrix dataframe
# Sorting columns naturally (S1, S2... S15)
sorted_cpt_ids = sorted(cpt_coordinates.keys(), key=lambda x: [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', x)])
df_distances = pd.DataFrame(index=df_boreholes['Borehole_ID'])

# Calculate absolute Euclidean distances
for cpt_id in sorted_cpt_ids:
    cpt_x, cpt_y = cpt_coordinates[cpt_id]
    
    distances = []
    for _, bh in df_boreholes.iterrows():
        # Euclidean distance formula
        dist = math.sqrt((bh['X'] - cpt_x)**2 + (bh['Y'] - cpt_y)**2)
        distances.append(round(dist, 2)) # Rounding to 2 decimals for readability
        
    df_distances[cpt_id] = distances

# print(cpt_coordinates)

# Reset index to make 'Borehole_ID' the first explicit column
df_distances = df_distances.reset_index()

# Apply the styling only to the CPT columns (excluding 'Borehole_ID')
# Note: Use .map() for newer pandas versions (>= 2.0). 
# If you are on an older version of pandas and get an error, change .map to .applymap
styled_distances = df_distances.style.map(highlight_close_distances, subset=sorted_cpt_ids).format("{:.2f}", subset=sorted_cpt_ids)

# Display the styled dataframe directly in your Jupyter Notebook
print("If multiple points in a row it means it is within the pre-determined radius")
styled_distances

If multiple points in a row it means it is within the pre-determined radius


,Borehole_ID,S1,S2,S3,S3A,S4,S5,S6,S7,S8,S9,S10,S11,S12,S13,S14,S15
0,1,45.28,41.49,32.57,33.39,28.69,22.23,14.72,18.48,3.56,41.55,38.54,29.92,18.34,14.42,18.28,8.16
1,2,5.62,16.51,8.72,8.73,16.52,22.62,26.33,35.54,38.75,2.45,12.34,14.63,22.75,26.78,28.84,32.51
2,3,41.33,46.99,33.69,35.07,22.12,31.29,17.25,3.22,22.38,38.31,42.64,34.95,18.90,23.48,10.13,20.26
3,4,27.53,36.55,22.35,23.83,8.85,25.46,14.12,11.88,26.67,24.78,31.84,25.66,12.63,20.58,7.84,21.45
4,5,36.53,35.39,24.63,25.64,19.47,17.17,5.72,13.00,9.21,32.86,31.80,23.24,9.37,8.66,10.04,4.72
5,6,12.68,8.89,7.14,5.86,20.36,18.27,26.09,37.67,36.58,10.25,4.53,9.58,22.93,24.52,30.72,30.74
6,7,35.03,38.31,25.57,26.86,16.11,22.24,8.20,5.92,16.45,31.66,34.13,26.16,10.07,14.61,3.29,12.48
7,8,11.86,26.27,13.37,14.55,8.32,24.74,22.00,27.26,35.86,9.71,21.56,19.56,18.46,25.25,21.34,29.56
8,9,29.59,27.39,16.83,17.75,15.10,10.08,3.97,18.08,14.17,25.83,23.74,15.20,4.54,3.64,12.24,7.84
9,10,17.64,13.49,3.50,3.37,15.53,10.61,17.88,30.34,28.14,14.05,9.13,3.16,14.99,16.05,23.42,22.26


# Zwolle

In [3]:
# Borehole data Zwolle
borehole_data = {
    'Borehole_ID': [
        25, 26, 52, 55, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
        14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 27, 28, 29, 30, 31, 32,
        33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
        51, 53, 54, 56, 57, 58, 59, 60
    ],
    'X': [
        202705, 202722, 202739, 202746, 202706, 202678, 202676, 202697, 202668, 202670, 202664, 202672, 202688, 202689, 202691, 202690, 202678,
        202677, 202687, 202667, 202673, 202664, 202680, 202677, 202695, 202683, 202692, 202699, 202695, 202704, 202712, 202722, 202716, 202708,
        202723, 202744, 202738, 202731, 202727, 202715, 202737, 202702, 202711, 202731, 202712, 202718, 202726, 202719, 202733, 202739, 202747, 202743,
        202740, 202698, 202704, 202729, 202731, 202738, 202683, 202705
    ],
    'Y': [
        503448, 503439, 503438, 503432, 503455, 503452, 503461, 503444, 503457, 503441, 503448, 503447, 503449, 503457, 503443, 503436, 503443,
        503435, 503431, 503434, 503428, 503428, 503429, 503420, 503424, 503436, 503417, 503435, 503452, 503440, 503438, 503431, 503449, 503432,
        503449, 503443, 503449, 503448, 503442, 503430, 503432, 503428, 503423, 503419, 503409, 503419, 503412, 503405, 503411, 503419, 503421, 503412,
        503426, 503415, 503420, 503433, 503425, 503444, 503421, 503409
    ]
}

df_boreholes = pd.DataFrame(borehole_data)

# Change this to the path where your .gef files are stored
gef_folder_path = r"C:\AA_Thesis\CPT_Measurements\Zwolle"

cpt_coordinates = {}

# Process each file in the folder
for filename in os.listdir(gef_folder_path):
    if filename.lower().endswith('.gef'): # make sure indifference for GEF or gef
        # Extract CPT ID using regex (matches everything between the last '_' and '.gef')
        match_id = re.search(r'_([A-Za-z0-9]+)\.gef$', filename, re.IGNORECASE) 
        if not match_id:
            continue
        cpt_id = match_id.group(1)
        
        filepath = os.path.join(gef_folder_path, filename)
        
        # Read file to find the #XYID line
        with open(filepath, 'r', encoding='latin-1') as file:
            for line in file:
                if line.startswith('#XYID='):
                    # Clean line and split by commas
                    parts = line.replace('#XYID=', '').strip().split(',')
                    if len(parts) >= 3:
                        try:
                            # 2nd value is X, 3rd value is Y
                            cpt_x = float(parts[1].strip())
                            cpt_y = float(parts[2].strip())
                            cpt_coordinates[cpt_id] = (cpt_x, cpt_y)
                        except ValueError:
                            print(f"Could not parse coordinates in file: {filename}")
                    break

# Initialize the final distance matrix dataframe
# Sorting columns naturally (S1, S2... S15)
sorted_cpt_ids = sorted(cpt_coordinates.keys(), key=lambda x: [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', x)])
df_distances = pd.DataFrame(index=df_boreholes['Borehole_ID'])

# Calculate absolute Euclidean distances
for cpt_id in sorted_cpt_ids:
    cpt_x, cpt_y = cpt_coordinates[cpt_id]
    
    distances = []
    for _, bh in df_boreholes.iterrows():
        # Euclidean distance formula
        dist = math.sqrt((bh['X'] - cpt_x)**2 + (bh['Y'] - cpt_y)**2)
        distances.append(round(dist, 2)) # Rounding to 2 decimals for readability
        
    df_distances[cpt_id] = distances

print(cpt_coordinates)

# Reset index to make 'Borehole_ID' the first explicit column
df_distances = df_distances.reset_index()

# Apply the styling only to the CPT columns (excluding 'Borehole_ID')
# Note: Use .map() for newer pandas versions (>= 2.0). 
# If you are on an older version of pandas and get an error, change .map to .applymap
styled_distances = df_distances.style.map(highlight_close_distances, subset=sorted_cpt_ids).format("{:.2f}", subset=sorted_cpt_ids)

# Display the styled dataframe directly in your Jupyter Notebook
print("If multiple points in a row it means it is within the pre-determined radius")
styled_distances

{'1': (202704.49, 503448.81), '10': (202739.16, 503437.83), '11': (202745.25, 503435.22), '112': (202669.16, 503435.28), '113': (202685.84, 503428.98), '114': (202709.19, 503421.52), '115': (202733.11, 503420.24), '12': (202745.75, 503431.54), '13': (202704.17, 503450.13), '14': (202706.99, 503448.89), '15': (202721.83, 503441.36), '16': (202721.81, 503437.55), '17': (202739.17, 503440.35), '18': (202739.17, 503436.56), '19': (202745.75, 503434.07), '2': (202721.83, 503440.1), '20': (202745.76, 503430.26), '201': (202667.94, 503432.33), '202': (202694.41, 503421.96), '203': (202730.2, 503427.65), '204': (202744.78, 503434.4), '205': (202738.85, 503446.48), '206': (202720.28, 503438.38), '207': (202702.43, 503435.17), '208': (202706.51, 503450.49), '209': (202690.32, 503455.03), '210': (202681.24, 503443.98), '3': (202739.17, 503439.11), '4': (202745.75, 503432.82), '5': (202704.5, 503451.35), '6': (202705.55, 503449.76), '7': (202721.84, 503442.66), '8': (202721.83, 503438.83), '9': (2

,Borehole_ID,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,112,113,114,115,201,202,203,204,205,206,207,208,209,210
0,25,0.96,18.59,35.31,43.49,3.39,1.84,17.67,19.17,34.76,35.64,42.23,43.95,2.29,2.18,18.09,19.79,35.02,36.03,43.07,44.45,38.03,27.00,26.81,39.51,40.24,28.11,32.39,42.04,33.88,18.06,13.08,2.91,16.28,24.10
1,26,20.07,1.11,17.17,24.54,21.42,19.66,3.66,0.24,17.39,17.20,23.56,24.89,21.02,17.98,2.37,1.46,17.22,17.34,24.26,25.32,52.97,37.52,21.67,21.80,54.47,32.43,14.00,23.24,18.44,1.83,19.94,19.29,35.50,41.06
2,52,36.16,17.30,1.12,8.51,36.99,35.46,17.78,17.19,3.67,0.23,6.84,9.34,36.88,33.81,17.50,17.20,2.36,1.45,7.81,10.28,69.89,53.92,34.06,18.71,71.29,47.39,13.59,6.81,8.48,18.72,36.68,34.81,51.57,58.07
3,55,44.78,25.49,9.86,0.86,45.79,44.18,26.41,25.12,11.83,8.99,3.31,0.52,45.59,42.51,25.92,24.82,10.79,8.21,2.09,1.76,76.91,60.24,38.27,17.45,78.06,52.56,16.39,2.69,16.15,26.50,43.69,43.60,60.25,65.86
4,1,6.37,21.74,36.78,45.52,3.95,5.26,20.08,22.63,35.76,37.34,43.95,46.16,5.20,6.19,20.90,23.55,36.26,37.95,44.92,46.83,41.79,32.92,33.63,44.08,44.30,35.01,36.52,43.91,33.94,21.91,20.15,4.54,15.68,27.10
5,2,26.68,45.42,62.51,70.41,26.51,27.64,44.82,45.77,62.05,62.78,69.31,70.77,26.24,29.16,45.10,46.13,62.27,63.09,70.08,71.16,18.91,24.32,43.61,63.61,22.09,34.23,57.60,69.06,61.10,44.42,29.67,28.55,12.69,8.65
6,3,30.99,50.37,66.86,75.23,30.09,31.62,49.37,50.91,66.07,67.28,73.89,75.72,30.19,33.27,49.86,51.46,66.46,67.73,74.77,76.23,26.61,33.50,51.58,70.16,29.78,43.16,63.64,73.74,64.51,49.72,36.96,32.27,15.51,17.81
7,4,8.90,25.13,42.45,50.02,10.50,10.31,24.88,25.36,42.24,42.61,49.04,50.32,9.43,11.12,24.97,25.63,42.33,42.82,49.75,50.66,29.17,18.71,25.57,43.23,31.32,22.19,37.01,48.73,41.92,23.95,10.37,11.51,12.90,15.76
8,5,37.40,56.42,73.38,81.42,36.93,38.24,55.72,56.81,72.81,73.70,80.26,81.81,36.82,39.82,56.06,57.22,73.09,74.05,81.06,82.23,21.75,33.22,54.36,74.77,24.67,43.88,68.78,80.04,71.63,55.50,40.77,39.06,22.41,18.57
9,6,35.36,51.84,69.20,76.19,36.02,36.61,51.87,51.88,69.18,69.23,75.47,76.34,35.37,37.82,51.83,51.92,69.17,69.31,76.07,76.52,5.78,19.88,43.76,66.44,8.91,30.96,61.66,75.07,69.07,50.35,32.95,37.72,24.69,11.63


## Map with coordinates

In [10]:
# =============================================================================
# USER SETTINGS
# =============================================================================

# Root folder containing your GEF files
ROOT_FOLDER = Path(r"C:\AA_Thesis\CPT_Measurements\Zwolle")

# Search pattern for GEF files
GEF_PATTERNS = ["*.gef", "*.GEF"]

# Output file for Google My Maps
OUTPUT_CSV = ROOT_FOLDER / "cpt_borehole_google_mymaps.csv"

# Coordinate transformation:
# Dutch RD New -> WGS84 lat/lon
transformer = Transformer.from_crs("EPSG:28992", "EPSG:4326", always_xy=True)


def parse_float(value):
    try:
        return float(str(value).strip().replace(",", "."))
    except Exception:
        return None


def read_xyid_from_gef(gef_path: Path):
    """
    Read X/Y coordinates from a GEF file.

    Expected form:
        #XYID= coordinate_system, x, y

    Returns:
        x_rd, y_rd
    """
    try:
        text = gef_path.read_text(encoding="ISO-8859-1", errors="replace")
    except Exception as exc:
        print(f"Could not read {gef_path}: {exc}")
        return None, None

    for line in text.splitlines():
        line = line.strip()

        if line.upper().startswith("#XYID="):
            raw_value = line.split("=", 1)[1]
            parts = [p.strip() for p in raw_value.split(",")]

            if len(parts) >= 3:
                x_rd = parse_float(parts[1])
                y_rd = parse_float(parts[2])
                return x_rd, y_rd

    return None, None


def find_gef_files(root_folder: Path):
    gef_files = []

    for pattern in GEF_PATTERNS:
        gef_files.extend(root_folder.rglob(pattern))

    return sorted(set(gef_files))


# =============================================================================
# READ GEF COORDINATES
# =============================================================================

gef_rows = []

for gef_path in find_gef_files(ROOT_FOLDER):
    x_rd, y_rd = read_xyid_from_gef(gef_path)

    if x_rd is None or y_rd is None:
        print(f"No coordinates found in: {gef_path}")
        continue

    lon, lat = transformer.transform(x_rd, y_rd)

    gef_rows.append({
        "label": gef_path.stem,
        "type": "CPT",
        "latitude": lat,
        "longitude": lon,
        "x_rd": x_rd,
        "y_rd": y_rd,
        "file_name": gef_path.name,
        "folder": str(gef_path.parent),
        "description": f"CPT | {gef_path.name} | RD: {x_rd:.2f}, {y_rd:.2f}",
    })

gef_df = pd.DataFrame(gef_rows)

print(f"Found {len(gef_df)} GEF files with coordinates.")


# =============================================================================
# MANUAL BOREHOLE COORDINATES
# =============================================================================
# x_rd and y_rd are Dutch RD coordinates.

BOREHOLES_RD = {
    "B25": {"x_rd": 202705, "y_rd": 503448},
    "B26": {"x_rd": 202722, "y_rd": 503439},
    "B52": {"x_rd": 202739, "y_rd": 503438},
    "B55": {"x_rd": 202746, "y_rd": 503432},
}

borehole_rows = []

for name, coords in BOREHOLES_RD.items():
    x_rd = float(coords["x_rd"])
    y_rd = float(coords["y_rd"])

    lon, lat = transformer.transform(x_rd, y_rd)

    borehole_rows.append({
        "label": name,
        "type": "Borehole",
        "latitude": lat,
        "longitude": lon,
        "x_rd": x_rd,
        "y_rd": y_rd,
        "file_name": "",
        "folder": "",
        "description": f"Borehole {name} | RD: {x_rd:.2f}, {y_rd:.2f}",
    })

borehole_df = pd.DataFrame(borehole_rows)


# =============================================================================
# COMBINE AND EXPORT FOR GOOGLE MY MAPS
# =============================================================================

all_points_df = pd.concat(
    [gef_df, borehole_df],
    ignore_index=True,
)

# My Maps-friendly column order
all_points_df = all_points_df[
    [
        "label",
        "type",
        "latitude",
        "longitude",
        "x_rd",
        "y_rd",
        "file_name",
        "folder",
        "description",
    ]
]

# Save for Google My Maps
all_points_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("")
print("Saved Google My Maps CSV to:")
print(OUTPUT_CSV)

print("")
print("For Google My Maps import:")
print("Location columns: latitude and longitude")
print("Marker title column: label")
print("Style by column: type")

display(all_points_df)

Found 34 GEF files with coordinates.

Saved Google My Maps CSV to:
C:\AA_Thesis\CPT_Measurements\Zwolle\cpt_borehole_google_mymaps.csv

For Google My Maps import:
Location columns: latitude and longitude
Marker title column: label
Style by column: type


,label,type,latitude,longitude,x_rd,y_rd,file_name,folder,description
0,61251107_1,CPT,52.516628,6.089998,202704.49,503448.81,61251107_1.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_1.GEF | RD: 202704.49, 503448.81"
1,61251107_10,CPT,52.516527,6.090507,202739.16,503437.83,61251107_10.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_10.GEF | RD: 202739.16, 503437.83"
2,61251107_11,CPT,52.516503,6.090596,202745.25,503435.22,61251107_11.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_11.GEF | RD: 202745.25, 503435.22"
3,61251107_112,CPT,52.516510,6.089475,202669.16,503435.28,61251107_112.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_112.GEF | RD: 202669.16, 503435.28"
4,61251107_113,CPT,52.516452,6.089720,202685.84,503428.98,61251107_113.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_113.GEF | RD: 202685.84, 503428.98"
5,61251107_114,CPT,52.516383,6.090063,202709.19,503421.52,61251107_114.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_114.GEF | RD: 202709.19, 503421.52"
6,61251107_115,CPT,52.516369,6.090415,202733.11,503420.24,61251107_115.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_115.GEF | RD: 202733.11, 503420.24"
7,61251107_12,CPT,52.516469,6.090603,202745.75,503431.54,61251107_12.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_12.GEF | RD: 202745.75, 503431.54"
8,61251107_13,CPT,52.516640,6.089993,202704.17,503450.13,61251107_13.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_13.GEF | RD: 202704.17, 503450.13"
9,61251107_14,CPT,52.516629,6.090034,202706.99,503448.89,61251107_14.GEF,C:\AA_Thesis\CPT_Measurements\Zwolle,"CPT | 61251107_14.GEF | RD: 202706.99, 503448.89"
